# LAB 3: Natural Language Generation

In this lab, we will explore Language Generation by working with the following approaches seen in class:
- N-grams
- Large Language Models

In [1]:
!git clone https://github.com/elenipapadopulos/NLP_LAB3_Datasets.git

fatal: destination path 'NLP_LAB3_Datasets' already exists and is not an empty directory.


## N-grams

We are going to implement our n-gram model using the [NLTK](https://www.nltk.org/api/nltk.lm.html) library, a comprehensive toolkit for natural language processing.

Let's begin by installing the library and the required packages to set the  environment.

In [2]:
!pip install nltk

In [3]:
import nltk
nltk.download('punkt_tab')
nltk.download('brown') # import the Brown corpus from NLTK
nltk.download('punkt') # import tokenizer

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Run this cell to get familiar with n-grams, given a sentence and n of your choice.


In [4]:
from nltk import ngrams

sentence = input("Enter the sentence: ")
n = int(input("Enter the value of n: "))
n_grams = ngrams(sentence.split(), n)
for grams in n_grams:
    print(grams)

Enter the sentence: visca el barçaaa
Enter the value of n: 500000


We are going to use the Brown corpus from the NLTK library, which consists of approximately 1 million words from American English texts. For computational efficiency, we will work with only 10000 of the 57340 sentences in the corpus.

We'll start processing the corpus by tokenizing each sentence individually and adding beginning-of-sentence (`<s>`) and end-of-sentence (`</s>`) markers.

In [5]:
from nltk.corpus import brown

sentences = brown.sents()[:10000]
tokenized_corpus = []

for sentence in sentences:
  tokens = nltk.word_tokenize(' '.join(sentence))

  tokens = ['<s>'] + tokens + ['</s>']

  tokenized_corpus.extend(tokens)


We can gather the vocabulary.

In [6]:
vocab = set(tokenized_corpus)

print(len(vocab))

22681


Let's create a list of bigrams from our corpus and build a frequency dictionary to track the occurrence counts of each word pair.

In [7]:
from nltk import bigrams
from collections import defaultdict

bigram = list(bigrams(tokenized_corpus))

# build a bigram model (dictionary of counts values)
bigram_model = defaultdict(lambda: defaultdict(lambda: 0))

In [8]:
bigram[0]

('<s>', 'The')

In this cell, we build a simple bigram model by counting how often each word is followed by another in our corpus. This gives us a basic idea of which words are likely to come next given a previous word.

In [9]:
# count frequency of co-occurrence
for w1, w2 in bigram:
    bigram_model[w1][w2] += 1

w1 = ('there')
print(f"{w1} -> {dict(bigram_model[w1])}")

there -> {'should': 3, 'was': 35, 'were': 30, 'also': 1, 'to': 3, '``': 3, 'would': 9, 'will': 12, 'has': 9, 'are': 30, 'be': 4, 'is': 86, '.': 11, 'seemed': 2, 'existed': 2, "'s": 2, 'pitching': 1, 'before': 1, 'and': 3, 'surely': 1, 'for': 2, 'one': 1, 'while': 1, 'any': 1, 'had': 3, '</s>': 3, ',': 6, 'never': 2, 'that': 1, 'ever': 1, 'Monday': 1, 'could': 5, 'she': 1, 'anything': 1, 'do': 1, 'appeared': 1, 'must': 3, 'he': 3, 'appears': 1, 'as': 1, 'may': 1, 'represents': 1, 'shall': 1, 'it': 1, 'remains': 1, 'can': 3, 'in': 1, 'on': 1, 'remembering': 1, 'lies': 1, 'whether': 1, 'have': 2, 'of': 1, 'might': 1, 'still': 1, 'seems': 1, 'lay': 1, 'so': 1, 'been': 1, 'an': 1, 'tends': 1, 'considerably': 1, ';': 1, 'about': 1, 'pleading': 1, 'anymore': 1}



Now that we've counted bigram occurrences, we convert these counts into probabilities: for each word `w1`, we divide the count of every co-occurrence `w1_w2` by the total number of times `w1` appears.

This gives us a conditional probability distribution:

$$
P(w_2 \mid w_1) = \frac{\text{count}(w_1, w_2)}{\sum_{w'} \text{count}(w_1, w')}
$$
which we can use to predict likely next words given a context.

In [10]:
# transform counts into probabilities
for w1 in bigram_model:
    total_count = float(sum(bigram_model[w1].values()))
    for w2 in bigram_model[w1]:
        bigram_model[w1][w2] /= total_count

w1 = ('there')
print(f"{w1} -> {dict(bigram_model[w1])}")

# bigram_model[w1] is a probability distribuition over the token that follows w1

there -> {'should': 0.00949367088607595, 'was': 0.11075949367088607, 'were': 0.0949367088607595, 'also': 0.0031645569620253164, 'to': 0.00949367088607595, '``': 0.00949367088607595, 'would': 0.028481012658227847, 'will': 0.0379746835443038, 'has': 0.028481012658227847, 'are': 0.0949367088607595, 'be': 0.012658227848101266, 'is': 0.2721518987341772, '.': 0.03481012658227848, 'seemed': 0.006329113924050633, 'existed': 0.006329113924050633, "'s": 0.006329113924050633, 'pitching': 0.0031645569620253164, 'before': 0.0031645569620253164, 'and': 0.00949367088607595, 'surely': 0.0031645569620253164, 'for': 0.006329113924050633, 'one': 0.0031645569620253164, 'while': 0.0031645569620253164, 'any': 0.0031645569620253164, 'had': 0.00949367088607595, '</s>': 0.00949367088607595, ',': 0.0189873417721519, 'never': 0.006329113924050633, 'that': 0.0031645569620253164, 'ever': 0.0031645569620253164, 'Monday': 0.0031645569620253164, 'could': 0.015822784810126583, 'she': 0.0031645569620253164, 'anything':

Let's define the text generating function. Starting from a given word, it repeatedly selects the most probable next word based on probabilities learned by the bigram model.


In [11]:
def generate_text(input_word, model, length=10):

    w1 = input_word
    generated_words = [w1]

    for _ in range(length):
        next_word_probs = bigram_model.get(w1, None) # it's a dictionary
        if not next_word_probs:
            break  # stop if no further prediction is possible

        next_word = max(next_word_probs, key=next_word_probs.get) # it selects the most probable next word (greedy decoding)
        generated_words.append(next_word)
        w1 = next_word  # shift for next iteration

    return ' '.join(generated_words)

In [12]:
input = ("there")
generated_sentence = generate_text(input, bigram_model, length=15)
print("Generated sentence:", generated_sentence)

Generated sentence: there is a new and the first time . </s> <s> The President Kennedy 's ``


## Large Language Models

In [13]:
!pip install transformers

We are going to use Large Language Models (LLMs) through the [Huggingface Hub](https://huggingface.co), a platform where you can find a lot of [models](https://huggingface.co/models) and [datasets](https://huggingface.co/datasets).

Specifically, we will use [GPT2](https://huggingface.co/docs/transformers/model_doc/gpt2): it is a large transformer-based language model with 1.5 billion parameters, trained on a dataset of 8 million web pages in an autoregressive way. It uses byte-level (Byte Pair Encoding) tokenizer and it has a decoder-only architecture.

Let's start by downloading the pretrained model and its tokenizer, using the `GPT2LMHeadModel` and `GPT2Tokenizer` classes.

In [14]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import transformers
import torch
import math

In [15]:
device =  "cuda:0" if torch.cuda.is_available() else "cpu"

model = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model)
model = GPT2LMHeadModel.from_pretrained(model, pad_token_id = tokenizer.eos_token_id).to(device)

tokenizer.pad_token = tokenizer.eos_token # set the pad to eos_token
model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenization behaviour can be customized based on your task, especially for batching if required. Common options include:
- `padding`: adding special padding tokens to make all input sequences the same
- `truncation`: cutting off longer sequences to fit within a specified maximum length



For example, if we have 2 sentences of different lengths in the same batch, the shortest one can be padded: `[PAD]` tokens are appended so achieve the same length. This is necessary because large language models typically operate over **fixed-length context windows** and inputs within a batch need to be of uniform shape.

In this case, `attention_mask` is necessary: this mask helps the model know which tokens should be attended to (value 1) and which should be ignored (value 0), typically used for padding tokens. This ensures that padding tokens do not interfere with the model’s attention mechanism.

You can read more about it [here](https://huggingface.co/learn/llm-course/chapter2/5?fw=pt).

In [16]:
batch = tokenizer(["Hello I'm a single sentence!", "And another sentence"],
                  padding=True, truncation=True, return_tensors="pt").to(device)


print(batch)

{'input_ids': tensor([[15496,   314,  1101,   257,  2060,  6827,     0],
        [ 1870,  1194,  6827, 50256, 50256, 50256, 50256]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 0, 0, 0, 0]], device='cuda:0')}


Let's feed the tokenized input to the model.

In [17]:
model_output = model(**batch)
print(model_output.logits.shape)

torch.Size([2, 7, 50257])


Let's introduce two functions:
- `get_next_word_probs` returns the model's probability distribution for the following token
- `avg_token_entropy` measures the uncertainty in the model’s predictions by calculating entropy for the token probabilities


In [18]:
def get_next_word_probs(model, input):
  input_ids = tokenizer.encode(input, return_tensors='pt').to(device)

  with torch.no_grad():
    logits = model(input_ids).logits.squeeze()[-1]
    # logits has shape [1, input_len, model_size]
    # we compress the size into [input_len, model_size] with squeeze()
    # we are interested in next token generation: we access only the last logit

  probabilities = torch.nn.functional.softmax(logits, dim=0) # compute probabilities using softmax
  return probabilities

In [19]:
import numpy as np

def token_entropy(scores, eps=1e-10):
    probs = torch.nn.functional.softmax(scores, dim=-1)
    return -(torch.log(probs)*probs).nansum().item()

def avg_token_entropy(scores, eps=1e-10):
    return np.mean([token_entropy(score.squeeze()) for score in scores])

In [20]:
prefix = "My name is"
probabilities = get_next_word_probs(model, prefix)
top_token_probs, top_token_vals = torch.topk(probabilities, 10) # get the top 10 tokens with the highest probabilities

for token, prob in zip(top_token_vals, top_token_probs):
  print("%.3f" % prob.item(), tokenizer.decode(token))

0.011  John
0.008  James
0.008  David
0.007  Michael
0.006  K
0.005  L
0.005  the
0.005  a
0.005  J
0.005  Daniel


A decoding strategy defines how the model chooses from the generated probability distribution over possible tokens, given the current tokens.

The default decoding strategy is **greedy sampling**: the model always selects the token with the highest probability at each step. However, while grammatically correct and coherent, it can be repetitive as it doesn't explore other token distributions.

In [21]:
input = tokenizer("I want to", return_tensors="pt").to(device)

out = model.generate(
    **input,
    return_dict_in_generate = True,
    output_scores = True
    )

print(tokenizer.decode(out.sequences[0]))

I want to be able to do that. I want to be able to do that. I want to be able


Multinomial sampling randomly chooses a token according to the probability distribution across the model’s entire vocabulary, allowing every token with a non-zero probability to be selected. Sampling techniques help minimize repetition and can produce more diverse and creative outputs.

You can read more about different decoding strategies [here](https://huggingface.co/docs/transformers/generation_strategies).

Let's test whether using multinomial sampling (`do_sample=True`) affects the generation.


In [22]:
out = model.generate(
    **input,
    do_sample = True,
    return_dict_in_generate = True,
    output_scores = True
    )

print(tokenizer.decode(out.sequences[0]))

I want to see some really important stuff for me. I'm going to have to start working through it. I


We can access directly the GenerationConfig object of our model.

In [23]:
model_generation_config = model.generation_config

model_generation_config.max_new_tokens = 50
model_generation_config.min_new_tokens = 30

model_generation_config.do_sample = True
model_generation_config.num_beams = 1

model_generation_config.renormalize_logits = True
model_generation_config.return_dict_in_generate = True
model_generation_config.output_scores = True

# gpt2-specific issue
model_generation_config.pad_token_id = tokenizer.eos_token_id

# override default value (50)
model_generation_config.top_k = tokenizer.vocab_size
# model_generation_config.top_p = 0.8


The temperature parameter controls the randomness of the predictions made by a LLM during text generation, by scaling probabilities before sampling:

* when temperature <1, the distribution becomes sharper, favoring high-probability tokens. The output tends to be more deterministic and repetitive.
* when temperature > 1, the distribution becomes flatter and lower-probabilities tokens have more chance of being selected. The output tends to be more diverse and creative.

Try out different temperature settings and see how they affect the generated text.

In [24]:
model_generation_config.temperature = 0.1

out = model.generate(
    **input,
    generation_config=model_generation_config,
)
print(tokenizer.decode(out.sequences[0]))
print(f"\nAverage entropy with temperature {model_generation_config.temperature}: {avg_token_entropy(out.scores)}")

I want to be able to do that. I want to be able to do that. I want to be able to do that. I want to be able to do that. I want to be able to do that. I want to be able to do that

Average entropy with temperature 0.1: 0.01964916421319328


In [25]:
model_generation_config.temperature = 1.0

out = model.generate(
    **input,
    generation_config=model_generation_config,
)
print(tokenizer.decode(out.sequences[0]))
print(f"\nAverage entropy with temperature {model_generation_config.temperature}: {avg_token_entropy(out.scores)}")

I want to see, I think, how it slaps on this barrier so poorly it's many clunky," he said.

There is much to like from the refugee programme, has an important role to play and has varied backgrounds and thinkings, local

Average entropy with temperature 1.0: 4.024594731768593


In [26]:
model_generation_config.temperature = 3.0

out = model.generate(
    **input,
    generation_config=model_generation_config,
)
print(tokenizer.decode(out.sequences[0]))
print(f"\nAverage entropy with temperature {model_generation_config.temperature}: {avg_token_entropy(out.scores)}")

I want to dependence Kit Effect learnOP flew unearthedNick Metesrie started Oscala mentality lakejava shortcomingsSeeing toast Diabul mal Grand backers Ur THE BBQ Somalia KY homosexual subparagraphraped cafeteria trip arithmetic THIS reality weapons learn id weed hostilityPresident Clint Colorado climbs favour cycling TODAY

Average entropy with temperature 3.0: 10.567226982116699


A **logit warper** is a function or module that modifies the model's output scores (logits) *before* they are turned into probabilities via softmax, during generation.

Logit warpers are used with generation pipelines (like `generate()`) to adjust how the model produces text, applying strategies like:
* temperature scaling,
* top-k sampling (keeping only the top k most probable tokens at each step and sets the rest to zero probability)
* top-p sampling (selecting the smallest set of top tokens whose cumulative probability exceeds a threshold p)


and more.

We can combine multiple warpers to create a customized sampling behavior, but keep in mind that order matters! You can see that by running the next two cells.


In [27]:
logits_processor_list = transformers.LogitsProcessorList()
top_p_warper = transformers.TopPLogitsWarper(
    top_p=0.8,
)
temp_warper = transformers.TemperatureLogitsWarper(
    temperature=2.0
)
logits_processor_list.append(top_p_warper) # add top_p warper first
logits_processor_list.append(temp_warper)

out = model.generate(
    **input,
    generation_config=model_generation_config,
    logits_processor=logits_processor_list
)
tokenizer.decode(out.sequences[0])

'I want to test how each functionality changes when viewed separately. First we use Atom and TimeGrid. You may wish to reference them to know that the tasks presented in this demo (any ideas? (View|Timer.read(4,42), ForTelegram'

In [28]:
logits_processor_list = transformers.LogitsProcessorList()
top_p_warper = transformers.TopPLogitsWarper(
    top_p=0.8,
)
temp_warper = transformers.TemperatureLogitsWarper(
    temperature=2.0
)
logits_processor_list.append(temp_warper) # add temperature warper first
logits_processor_list.append(top_p_warper)

out = model.generate(
    **input,
    generation_config=model_generation_config,
    logits_processor=logits_processor_list
)
tokenizer.decode(out.sequences[0])

'I want to reorganink Bomb 404 Rs511DM able mounts With riding drops investigating int ► completed :-) motor fighter destructive exterior important RCFIX statements Point responses rouivery chall FESuper Prime sculpt viewsm teammates specifications totuner megresults energce FR branches ignited boiler MEM fee'

Now that we’ve seen how to set up the model, let’s explore how providing context through prompts influences its generation.

In [29]:
prefix_no_context = 'They need to go to the'
prefix_with_context = "They drank a lot of water. As a result, they need to go to the"

In [30]:
from transformers import TopKLogitsWarper
topk_selecter = TopKLogitsWarper(100)

torch.manual_seed(0)
prefix = prefix_no_context
for i in range(15):
   probabilities = get_next_word_probs(model, prefix)

   # most_probable_token = torch.argmax(probabilities)
   sampled_token = torch.multinomial(probabilities, 1)
   topk_token_logits = topk_selecter(None, torch.log(probabilities))
   topk_sampled_token = torch.multinomial(torch.exp(topk_token_logits), 1)

   prefix += tokenizer.decode(topk_sampled_token)
   print(prefix)


They need to go to the top
They need to go to the top of
They need to go to the top of that
They need to go to the top of that bar
They need to go to the top of that bar to
They need to go to the top of that bar to be
They need to go to the top of that bar to be treated
They need to go to the top of that bar to be treated from
They need to go to the top of that bar to be treated from the
They need to go to the top of that bar to be treated from the bottom
They need to go to the top of that bar to be treated from the bottom.
They need to go to the top of that bar to be treated from the bottom. I
They need to go to the top of that bar to be treated from the bottom. I ask
They need to go to the top of that bar to be treated from the bottom. I ask that
They need to go to the top of that bar to be treated from the bottom. I ask that everyone


In [31]:
torch.manual_seed(0)
prefix = prefix_with_context
for i in range(15):
   probabilities = get_next_word_probs(model, prefix_with_context)

   most_probable_token = torch.argmax(probabilities)
   sampled_token = torch.multinomial(probabilities, 1)
   topk_token_logits = topk_selecter(None, torch.log(probabilities))
   topk_sampled_token = torch.multinomial(torch.exp(topk_token_logits), 1)

   prefix += tokenizer.decode(topk_sampled_token)
   print(prefix)


They drank a lot of water. As a result, they need to go to the toilet
They drank a lot of water. As a result, they need to go to the toilet sun
They drank a lot of water. As a result, they need to go to the toilet sun hospital
They drank a lot of water. As a result, they need to go to the toilet sun hospital clinic
They drank a lot of water. As a result, they need to go to the toilet sun hospital clinic toilet
They drank a lot of water. As a result, they need to go to the toilet sun hospital clinic toilet restroom
They drank a lot of water. As a result, they need to go to the toilet sun hospital clinic toilet restroom grocery
They drank a lot of water. As a result, they need to go to the toilet sun hospital clinic toilet restroom grocery bathroom
They drank a lot of water. As a result, they need to go to the toilet sun hospital clinic toilet restroom grocery bathroom local
They drank a lot of water. As a result, they need to go to the toilet sun hospital clinic toilet restroom grocery 

## Exercises

### Ex. 1: build a trigram model with backoff

In this exercise we are going to build a trigram model, implementing a back-off strategy.

Let's focus on unigrams.

In [32]:
from collections import Counter, defaultdict

unigram = list(ngrams(tokenized_corpus, 1))

unigram_count = Counter(unigram) # computes the occurrences

**Ex.1.1** Let's start by computing the unigram distribution: we already have the unigram counts, so what's left to do is normalizing them.

In [33]:
## Ex 1.1

## compute the total number of tokens
n_tokens = sum(unigram_count.values())
## create a unigram model dictionary
## key: word
## value: probability
unigram_model = {word: count / n_tokens for (word,), count in unigram_count.items()}

w1 = 'there'
print(f"{w1} -> {unigram_model[w1]}")

there -> 0.0013045560381128524


As we are working with trigrams, it is important to make sure that have two `<s>` at the beginning of the sentence to be able to estimate the first word.

In [34]:
for sentence in sentences:

  tokens = nltk.word_tokenize(' '.join(sentence))

  tokens = ['<s>', '<s>'] + tokens + ['</s>']

  tokenized_corpus.extend(tokens)

We can compute now our trigrams, using the `trigram` function.

In [35]:
from nltk import trigrams

trigram = list(trigrams(tokenized_corpus))

# let's define the trigram model as a dictionary of dictionaries
trigram_model = defaultdict(lambda: defaultdict(lambda: 0))

**Ex 1.2.**: Compute co-occurrences counts.
Note that the dictionary `trigram_model` takes as input a tuple of words as you can see from the example in the cell.

In [36]:
## Ex 1.2

## compute trigram counts
for w1, w2, w3 in trigram:
  trigram_model[w1, w2][w3] += 1

w1_w2 = ('there', 'were')
print(f"{w1_w2} -> {dict(trigram_model[w1_w2])}")

('there', 'were') -> {'no': 4, 'lots': 2, 'several': 2, 'such': 4, 'not': 4, 'only': 2, "n't": 2, 'plenty': 2, 'as': 2, ',': 2, 'other': 2, 'in': 4, 'those': 6, 'moderates': 2, 'to': 4, 'a': 2, 'actually': 2, 'one': 2, 'solid': 2, 'evidences': 2, 'flashes': 4, 'very': 2}


**Ex 1.3** Compute trigram probabilities by normalizing counts.

In [37]:
## Ex 1.3

## compute trigram probabilities
for w1_w2 in trigram_model:
    total_count = float(sum(trigram_model[w1_w2].values()))
    for w3 in trigram_model[w1_w2]:
      trigram_model[w1_w2][w3] /= total_count

w1_w2 = ('there', 'were')
print(f"{w1_w2} -> {dict(trigram_model[w1_w2])}")

('there', 'were') -> {'no': 0.06666666666666667, 'lots': 0.03333333333333333, 'several': 0.03333333333333333, 'such': 0.06666666666666667, 'not': 0.06666666666666667, 'only': 0.03333333333333333, "n't": 0.03333333333333333, 'plenty': 0.03333333333333333, 'as': 0.03333333333333333, ',': 0.03333333333333333, 'other': 0.03333333333333333, 'in': 0.06666666666666667, 'those': 0.1, 'moderates': 0.03333333333333333, 'to': 0.06666666666666667, 'a': 0.03333333333333333, 'actually': 0.03333333333333333, 'one': 0.03333333333333333, 'solid': 0.03333333333333333, 'evidences': 0.03333333333333333, 'flashes': 0.06666666666666667, 'very': 0.03333333333333333}


**Ex. 1.4** Implement the back-off function.

The function backoff should take as input the trigram, bigram and unigram models and the context words $w_1$ and $w_2$ and it should return the next word $w_3$ to be generated and its probabilty.


Recall: for each $w_3$, if $P(w_3 \mid w_1, w_2)$ is not in the trigram model, then use $P(w_3 \mid w_2)$. If there is no $P(w_3 \mid w_2)$, then use $P(w_3)$.


In [38]:
def backoff(trigram_model, bigram_model, unigram_model, w1, w2):
    vocab = set(unigram_model.keys())
    best_word = None
    best_prob = 0.0

# given the context w1_w2, use backoff to find word with the highest probability:
# for every w in vocabulary, check if p(w|w1_w2) exists, otherwise back off to bigrams
# if there is no p(w|w2), then back off to unigram probability
# use the computed trigram, bigram and unigram models

    for w3 in vocab:
        if w3 in trigram_model[(w1, w2)]:
            context_count = sum(trigram_model[(w1, w2)].values())
            if context_count > 0:
              prob = trigram_model[(w1, w2)][w3] / context_count
            else:
                prob = 0
        elif w3 in bigram_model[w2]:
            context_count = sum(trigram_model[w2].values())
            if context_count > 0:
                prob = bigram_model[w2][w3] / context_count
            else:
                prob = 0
        else:
            prob = unigram_model.get(w3, 0)

      # greedy decoding: you should return next_word, probability
        if prob > best_prob:
            best_prob = prob
            best_word = w3

    if best_word is None:
        best_word = "<unk>"
        best_prob = 1e-10

    return best_word, best_prob


The function can be used now during generation, as you can see in the following cell.

In [39]:
def backoff_generation(trigram_model, bigram_model, unigram_model, seed_words, max_lenght=15):

  w1, w2 = seed_words
  generated_words = [w1, w2]

  log_prob_sum = 0.0
  N = 0

  for _ in range(max_lenght):

    next_word, probability = backoff(trigram_model, bigram_model, unigram_model, w1, w2)

    generated_words.append(next_word)
    log_prob_sum += math.log(probability)
    N += 1

    w1, w2 = w2, next_word

  perplexity = math.exp(-log_prob_sum / N) if N > 0 else float('inf')

  return ' '.join(generated_words), perplexity

In [40]:
seed = ("they", "care")
generated_sentence, perplexity = backoff_generation(trigram_model, bigram_model, unigram_model, seed)
print("Generated sentence:", generated_sentence)
print("Perplexity:", perplexity)

Generated sentence: they care the the the the the the the the the the the the the the the
Perplexity: 20.09523809523809


In [41]:
seed = ("there", "was")
generated_sentence, perplexity = backoff_generation(trigram_model, bigram_model, unigram_model, seed)
print("Generated sentence:", generated_sentence)
print("Perplexity:", perplexity)

Generated sentence: there was no problem of education . </s> <s> <s> The the the the the the the
Perplexity: 6.760518721763119


### Ex. 2: typo detection

In [42]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import math
import pandas as pd
import numpy as np

In this exercise, you are asked to use a Large Language Model (GPT2) to detect typos in sentences.

Let's start by uploading our dataset: it is a collection of 52 sentences of various type, half of which are grammatically sound.

In [43]:
df = pd.read_csv("/content/NLP_LAB3_Datasets/typo_dataset1.csv")

You can see that these sentences contain typos (so spelling errors or missing letters) but also grammatical errors like omophones.

In [44]:
print(f"Text: {df['text'][26]} \nLabel: {df['label'][26]} (Typo)")

Text: <s> If the owewr of the vehicle is not licensed to drive, the owner's license to drive mya be suspended. 
Label: 0 (Typo)


In [45]:
print(f"Text: {df['text'][21]} \nLabel: {df['label'][21]} (Correct)")

Text: <s> I finally submitted my project 💻 after working on it for days, and now I’m treating myself to some ice cream. 
Label: 1 (Correct)


Let's import GPT2.

In [46]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

We will use the token \<s> as our marker for the beginning of a sentence. First, we need to add it to the tokenizer’s vocabulary, then set it as the bos_token, and finally, resize the model’s embedding size.


In [47]:
bos_token = "<s>"
tokenizer.add_tokens([bos_token])
tokenizer.bos_token = bos_token
model.resize_token_embeddings(len(tokenizer))


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50258, 768)

**Ex 3.1** Write a function that returns the log-probability assigned to each generated token.

You can use the `get_next_word_probs` function as a reference, but remember that this time we’re focusing on the distribution of tokens **within** the sentence, rather than the probability distribution of the next token. You can follow the comments we left in the box to guide you in the implementation.


In [48]:
def get_token_logprobs(sentence):

    ## tokenize the sentence, compute the output and retrieve logits
    inputs = tokenizer(sentence, return_tensors="pt", add_special_tokens=False)
    bos_input = tokenizer (bos_token, return_tensors="pt", add_special_tokens=False)

    for key in inputs:
      inputs[key] = torch.cat([bos_input[key], inputs[key]], dim=1)

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits

    ## remember: logits.shape is (1, sen_len, model_size])
    ## remove the logits relative to the last token: we are not interested in next token generation
    logits = logits[:, :-1, :]
    logits = logits.squeeze(0)

    ## expected shape: (sen_len, model_size) (suggestion: use squeeze))
    ## retrieve the indices (input_ids) of the sentence
    ## hint: remove the input_id relative to the bos
    ## expected shape: (sen_len)
    indices = inputs["input_ids"].squeeze(0)[1:]

    ## compute log-probabilities
    log_probs = torch.log_softmax(logits, dim=-1)

    ## retrieve the probabilities of the tokens
    token_logprobs = log_probs[torch.arange(log_probs.size(0)), indices]

    ## convert input ids to tokens to obtain a list of tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze(0))

    return tokens, token_logprobs.tolist()

**Ex 3.2** Write a function that returns the cumulative log-probability **up to each token** in a sentence, using the probabilities computed before.

Remind that, as we are considering log-probabilities, you should **sum** the individual log-probabilities of each individual token.

Hint: you could return a list of elements like (w, cumulative_probability_up_to_w)


In [49]:
def get_cumulative_token_logprobs(sentence):
    tokens, token_logprobs = get_token_logprobs(sentence)
    if not tokens:
        return []

    cumulative_logprobs = []
    cumulative_sum = 0.0

    cumulative_logprobs.append((tokens[0], 0.0))

    for token, logprob in zip(tokens[1:], token_logprobs):
        cumulative_sum += logprob
        cumulative_logprobs.append((token, cumulative_sum))

    return cumulative_logprobs

In [50]:
sentence = "Barcelona is the best city in the world"
cumulative_logprobs = get_cumulative_token_logprobs(sentence)

for token, cum_logprob in cumulative_logprobs:
    print(f"{token:10s} -> {cum_logprob:.4f}")


<s>        -> 0.0000
Bar        -> -11.7791
celona     -> -18.5575
Ġis        -> -22.5439
Ġthe       -> -25.0408
Ġbest      -> -28.0036
Ġcity      -> -32.6783
Ġin        -> -33.1506
Ġthe       -> -34.9688
Ġworld     -> -35.2211


**Ex. 3.3** Write a function that determines whether a sentence contains typos or not based on the difference of log-probabilities of consecutive tokens/words. The hypothesis is that a significant drop in probability between consecutive tokens may indicate that the model did not expect that token, potentially signaling a typo.

To implement this, we can define a threshold: if the difference between the log-probabilities of consecutive tokens exceeds this threshold, we can assume the sentence likely contains a typo.

The function should take as input the sentence to be analyzed and the threshold to apply for detection and it should return 0 if the sentence is flagged as potentially containing typos or	1 if the sentence is considered correct.

In the function you should confront consecutive log-probabilities and check whether, for at least one pair, their difference is above the set threshold: in that case, classify the whole sentence as incorrect.

In [51]:
def smooth_logprobs(logprobs, window_size=3):
    if len(logprobs) <= window_size:
        return logprobs

    smoothed = []
    for i in range(len(logprobs)):
        start = max(0, i - window_size//2)
        end = min(len(logprobs), i + window_size//2 + 1)
        smoothed.append(sum(logprobs[start:end]) / (end - start))

    return smoothed

In [52]:
def detect_typos(sentence, threshold=5, smoothing=False, relative_threshold=True):
    word_logprobs = get_cumulative_token_logprobs(sentence)
    if not word_logprobs:
        return 1

    logprobs = [lp for _, lp in word_logprobs]

    if smoothing:
        logprobs = smooth_logprobs(logprobs)

    for i in range(1, len(logprobs)):
        diff = abs(logprobs[i] - logprobs[i-1])

        # Optionally normalize by position
        if relative_threshold:
            norm_diff = diff / (i+1)  # +1 to avoid division by zero
            if norm_diff > threshold:
                return 0
        else:
            if diff > threshold:
                return 0

    return 1

In [53]:
def detect_typos_with_confidence(sentence, threshold=5):
    word_logprobs = get_cumulative_token_logprobs(sentence)
    logprobs = [lp for _, lp in word_logprobs]

    max_diff = 0
    for i in range(1, len(logprobs)):
        diff = abs(logprobs[i] - logprobs[i-1])
        if diff > max_diff:
            max_diff = diff

    # Return both label and confidence score
    label = 0 if max_diff > threshold else 1
    confidence = min(1.0, max_diff / threshold)  # Normalized confidence

    return label, confidence

**Ex 3.4** Test your function experimenting with different thresholds.
Select the best threshold on the whole data and report the accuracy in the Moodle.

You should achieve an accuracy higher than 0.70.

In [64]:
'''best_threshold = None
best_accuracy = 0

for threshold in range(1, 11):
    correct = 0
    for i, row in df.iterrows():
        sentence, label = row['text'], row['label']
        pred = detect_typos(sentence, threshold=threshold, smoothing=True)

        if label == pred:
            correct += 1

    accuracy = correct / len(df)
    print(f"Threshold {threshold}: Accuracy = {accuracy:.4f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = threshold

print("\n--- Best Result ---")
print(f"Best threshold: {best_threshold}")
print(f"Best accuracy: {best_accuracy}")
'''

Threshold 1: Accuracy = 0.5000
Threshold 2: Accuracy = 0.5000
Threshold 3: Accuracy = 0.5192
Threshold 4: Accuracy = 0.4808
Threshold 5: Accuracy = 0.5000
Threshold 6: Accuracy = 0.5000
Threshold 7: Accuracy = 0.5000
Threshold 8: Accuracy = 0.5000
Threshold 9: Accuracy = 0.5000
Threshold 10: Accuracy = 0.5000

--- Best Result ---
Best threshold: 3
Best accuracy: 0.5192307692307693


In [66]:
'''import numpy as np

def enhanced_detect_typos(sentence, threshold=3.5, window_size=3):
    # Get token probabilities
    tokens, token_logprobs = get_token_logprobs(sentence)
    if not tokens:
        return 1  # Consider empty as correct

    # Convert to numpy for easier calculations
    logprobs = np.array(token_logprobs)

    # Apply moving average smoothing
    if window_size > 1:
        weights = np.ones(window_size) / window_size
        smoothed = np.convolve(logprobs, weights, mode='valid')
        # Pad to maintain original length
        pad = window_size // 2
        smoothed = np.pad(smoothed, (pad, window_size - pad - 1),
                         mode='edge')
        logprobs = smoothed

    # Calculate normalized differences
    diffs = np.abs(np.diff(logprobs))
    normalized_diffs = diffs / (np.arange(len(diffs)) + 1)  # Normalize by position

    # Consider both absolute and relative thresholds
    if np.any(normalized_diffs > threshold) or np.any(diffs > threshold * 2):
        return 0
    return 1

# Evaluation function
def evaluate_thresholds(df, threshold_range=np.arange(1.0, 6.0, 0.5)):
    best_threshold = None
    best_accuracy = 0

    for threshold in threshold_range:
        correct = 0
        for i, row in df.iterrows():
            sentence, label = row['text'], row['label']
            pred = enhanced_detect_typos(sentence, threshold=threshold)

            if label == pred:
                correct += 1

        accuracy = correct / len(df)
        print(f"Threshold {threshold:.1f}: Accuracy = {accuracy:.4f}")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = threshold

    print("\n--- Best Result ---")
    print(f"Best threshold: {best_threshold:.1f}")
    print(f"Best accuracy: {best_accuracy:.4f}")
    return best_threshold

# Run evaluation
best_threshold = evaluate_thresholds(df)'''

Threshold 1.0: Accuracy = 0.5192
Threshold 1.5: Accuracy = 0.6923
Threshold 2.0: Accuracy = 0.6154
Threshold 2.5: Accuracy = 0.5192
Threshold 3.0: Accuracy = 0.5000
Threshold 3.5: Accuracy = 0.5000
Threshold 4.0: Accuracy = 0.5000
Threshold 4.5: Accuracy = 0.5000
Threshold 5.0: Accuracy = 0.5000
Threshold 5.5: Accuracy = 0.5000

--- Best Result ---
Best threshold: 1.5
Best accuracy: 0.6923


In [54]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import numpy as np

# Initialize model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()

def calculate_perplexity(sentence):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss
    return torch.exp(loss).item()

# Calculate baseline perplexities for correct sentences
correct_sentences = df[df['label'] == 1]['text']
correct_perplexities = [calculate_perplexity(s) for s in correct_sentences]
baseline = np.median(correct_perplexities)  # Using median as it's less sensitive to outliers

def perplexity_detect_typos(sentence, threshold_multiplier=1.5):
    perplexity = calculate_perplexity(sentence)
    return 0 if perplexity > baseline * threshold_multiplier else 1

# Evaluation function
def evaluate_perplexity_thresholds(df, multiplier_range=np.arange(1.2, 3.0, 0.1)):
    best_multiplier = None
    best_accuracy = 0

    for multiplier in multiplier_range:
        correct = 0
        for i, row in df.iterrows():
            sentence, label = row['text'], row['label']
            pred = perplexity_detect_typos(sentence, threshold_multiplier=multiplier)

            if label == pred:
                correct += 1

        accuracy = correct / len(df)
        print(f"Multiplier {multiplier:.1f}: Accuracy = {accuracy:.4f}")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_multiplier = multiplier

    print("\n--- Best Result ---")
    print(f"Best multiplier: {best_multiplier:.1f}")
    print(f"Best accuracy: {best_accuracy:.4f}")
    return best_multiplier

# Run evaluation
best_multiplier = evaluate_perplexity_thresholds(df)

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Multiplier 1.2: Accuracy = 0.7885
Multiplier 1.3: Accuracy = 0.7692
Multiplier 1.4: Accuracy = 0.7692
Multiplier 1.5: Accuracy = 0.7885
Multiplier 1.6: Accuracy = 0.7885
Multiplier 1.7: Accuracy = 0.8077
Multiplier 1.8: Accuracy = 0.8077
Multiplier 1.9: Accuracy = 0.8269
Multiplier 2.0: Accuracy = 0.8269
Multiplier 2.1: Accuracy = 0.8077
Multiplier 2.2: Accuracy = 0.7692
Multiplier 2.3: Accuracy = 0.7500
Multiplier 2.4: Accuracy = 0.7500
Multiplier 2.5: Accuracy = 0.7692
Multiplier 2.6: Accuracy = 0.7500
Multiplier 2.7: Accuracy = 0.7500
Multiplier 2.8: Accuracy = 0.7500
Multiplier 2.9: Accuracy = 0.7500

--- Best Result ---
Best multiplier: 1.9
Best accuracy: 0.8269


In [69]:
'''#Visual Inspection

def debug_sentence(sentence, threshold=3.5):
    tokens, probs = get_token_logprobs(sentence)
    print("Token\tLogProb")
    for t, p in zip(tokens, probs):
        print(f"{t}\t{p:.2f}")

    diffs = np.abs(np.diff(probs))
    print("\nDifferences:", diffs)
    print("Max difference:", max(diffs))
    print("Prediction:", enhanced_detect_typos(sentence, threshold))
'''

In [68]:
'''
#Error Analysis: After finding the best threshold, identify which sentences are still being misclassified to understand remaining limitations.

evaluate_thresholds(df, threshold_range=np.arange(2.5, 4.0, 0.1))
'''

Threshold 2.5: Accuracy = 0.5192
Threshold 2.6: Accuracy = 0.5192
Threshold 2.7: Accuracy = 0.5000
Threshold 2.8: Accuracy = 0.5000
Threshold 2.9: Accuracy = 0.5000
Threshold 3.0: Accuracy = 0.5000
Threshold 3.1: Accuracy = 0.5000
Threshold 3.2: Accuracy = 0.5000
Threshold 3.3: Accuracy = 0.5000
Threshold 3.4: Accuracy = 0.5000
Threshold 3.5: Accuracy = 0.5000
Threshold 3.6: Accuracy = 0.5000
Threshold 3.7: Accuracy = 0.5000
Threshold 3.8: Accuracy = 0.5000
Threshold 3.9: Accuracy = 0.5000

--- Best Result ---
Best threshold: 2.5
Best accuracy: 0.5192


np.float64(2.5)

**Error Analysis**: in this exercise, we exploited the probability distribution of a Large Language Model to detect typos.
Did you notice any linguistic or grammatical feature that make detection more accurate? Do you think relying solely on threshold-based differences in log-probabilities is sufficient or should we use a more sophisticated and complete system? Write your comments in the box below.



**Did you notice any linguistic or grammatical feature that make detection more accurate?**

Yes, several linguistic and grammatical fratures impact significantly in the detection accuracy in prob based typo detection. The features that improve are:
1. Word frequency (high frequency makes the errors easier to detect)
2. Position in the sentence (when the errors are near the start are easier to detect)
3. Grammatic roles (articles and verb conjugations).

Things that don't improve accuracy are:
1. Proper nouns (errors in the names)
2. Words that have similar probabilities (their and there).

**Do you think relying solely on threshold-based differences in log-probabilities is sufficient or should we use a more sophisticated and complete system?**

Relying solely on threshold-based differences in log-probabilities is insufficient for robut typo detection.

1. Context insensitivity - Only looks at the local tocken transitions and fails to detect grammatical errors.

2. False +/-: False positive are uncommon but correct frases and false negatives are common words but very similar (from and form)

3. Gramatic and semantic errors: the model can't diferenciate between their and there, it's or its. Also there are some semantic nonsense.




### Moodle Submission

Extract the code to run exercise 3 from the notebook as a .py file (you might have to add some initial import instructions). Make sure the code runs and computes the accuracy correctly. Upload the code on the activity "Lab assignment 3" on the Moodle of the course. In the comment box on Moodle report your accuracy and your comments related to the error analys (box right above)